In [ ]:
"""
Temporal BPR-MF with Time Decay
Variant: Temporal Model for Myket Android App Recommender

Reference:
Rendle et al. "BPR: Bayesian Personalized Ranking from Implicit Feedback" (UAI 2009)
https://arxiv.org/abs/1205.2618
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import load_npz
import joblib
from tqdm import tqdm


In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

CONFIG = {
    "embedding_dim": 64,              #size of user/item embeddings
    "batch_size":    4096,            #Large batches stabilize BPR
    "lr":            1e-3,            #Adam learning rate
    "weight_decay":  1e-5,
    "n_epochs":      10,
    "time_decay_halflife_days": 15,   # interactions older than this have half-weight
    "n_neg_samples": 1,
    "seed":          42,
}

#Fix random seeds so results are reproducible across rusn

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])


Using device: cpu


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. LOAD PIPELINE OUTPUTS
# ══════════════════════════════════════════════════════════════
# These are the artifacts produced by the shared Phase 1 pipeline.
# train.csv: interactions used for training
# val.csv:   per-user held-out interactions for hyperparameter tuning
# test.csv:  per-user held-out interactions for final evaluation
# user_train_history.pkl: dict mapping user_id → list of app_ids
train_df = pd.read_csv("pipeline_output/train.csv")
val_df   = pd.read_csv("pipeline_output/val.csv")
test_df  = pd.read_csv("pipeline_output/test.csv")
user_train_history = joblib.load("pipeline_output/user_train_history.pkl")

n_users = train_df["user_id"].max() + 1   # 10,000
n_items = train_df["app_id"].max() + 1    # 7,988

print(f"Users: {n_users} | Items: {n_items}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


Users: 10000 | Items: 7988
Train: 293,430 | Val: 45,205 | Test: 45,205


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. COMPUTE TIME DECAY WEIGHTS
#    w(t) = 0.5 ^ ((t_max - t) / halflife)
#    Recent interactions ≈ 1.0, older ones decay exponentially
# ══════════════════════════════════════════════════════════════
t_max = train_df["timestamp"].max()                               #timestamp of most recent interaction
halflife_seconds = CONFIG["time_decay_halflife_days"] * 86400

age_seconds = t_max - train_df["timestamp"].values                #Compute age in seconds for every interaction
train_df["weight"] = 0.5 ** (age_seconds / halflife_seconds)

print(f"\nTime weight stats:")
print(f"  min={train_df['weight'].min():.4f}  "
      f"max={train_df['weight'].max():.4f}  "
      f"mean={train_df['weight'].mean():.4f}")

# Build set of interacted items per user for efficient negative sampling
user_seen_items = {
    uid: set(items) for uid, items in user_train_history.items()
}



Time weight stats:
  min=0.0063  max=1.0000  mean=0.1753


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. BPR DATASET — samples (user, pos, neg) triplets with weight
# ══════════════════════════════════════════════════════════════
# Each training example is a triplet: (user u, positive item i, negative item j)
class BPRDataset(Dataset):
    def __init__(self, df, n_items, user_seen_items):
        self.users   = df["user_id"].values
        self.pos     = df["app_id"].values
        self.weights = df["weight"].values.astype(np.float32)
        self.n_items = n_items
        self.user_seen_items = user_seen_items

    def __len__(self):
      #Number of training interactions
        return len(self.users)

    def __getitem__(self, idx):
      #retrieve the positive interactions at this index
        u   = self.users[idx]
        pos = self.pos[idx]
        w   = self.weights[idx]

        # Sample a negative the user has NOT interacted with
        seen = self.user_seen_items.get(u, set())
        while True:
            neg = np.random.randint(self.n_items)
            if neg not in seen:
                break
        return u, pos, neg, w

train_ds = BPRDataset(train_df, n_items, user_seen_items)
train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. BPR-MF MODEL
# ══════════════════════════════════════════════════════════════
# The model has three trainable parts:
# user_emb:  one 64-dim vector per user (10,000 × 64 matrix)
# item_emb:  one 64-dim vector per item (7,988 × 64 matrix)
# item_bias: one scalar per item, captures global item popularity
class BPRMF(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.item_bias = nn.Embedding(n_items, 1)

        nn.init.normal_(self.user_emb.weight,  std=0.01)
        nn.init.normal_(self.item_emb.weight,  std=0.01)
        nn.init.zeros_(self.item_bias.weight)

    def score(self, u, i):
        """Compute score(u, i) = <p_u, q_i> + b_i"""
        return (self.user_emb(u) * self.item_emb(i)).sum(dim=1) + \
               self.item_bias(i).squeeze(-1)

    def forward(self, u, pos, neg):
        return self.score(u, pos), self.score(u, neg)

    def full_user_scores(self, u_ids):
        """Score user against ALL items — for inference."""
        u_vec = self.user_emb(u_ids)                    # (B, D)
        scores = u_vec @ self.item_emb.weight.T         # (B, n_items)
        scores = scores + self.item_bias.weight.squeeze(-1)  # broadcast
        return scores

model = BPRMF(n_users, n_items, CONFIG["embedding_dim"]).to(DEVICE)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. TRAIN LOOP — weighted BPR loss
#    L = -w * log(sigmoid(score_pos - score_neg))
# ══════════════════════════════════════════════════════════════
def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    n_batches  = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for u, pos, neg, w in pbar:
        u   = u.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)
        w   = w.to(DEVICE)

        s_pos, s_neg = model(u, pos, neg)
        # Weighted BPR loss: downweight old interactions
        loss = -(w * torch.log(torch.sigmoid(s_pos - s_neg) + 1e-10)).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / n_batches

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. EVALUATION — Recall@K and NDCG@K
# ══════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(eval_df, k_list=(5, 10, 20)):
    model.eval()
    # Group eval interactions per user → set of held-out positives
    user_positives = eval_df.groupby("user_id")["app_id"].apply(set).to_dict()

    metrics = {f"recall@{k}": [] for k in k_list}
    metrics.update({f"ndcg@{k}": [] for k in k_list})

    users = list(user_positives.keys())
    batch_size = 512

    for i in range(0, len(users), batch_size):
        batch_users = users[i:i + batch_size]
        u_tensor = torch.LongTensor(batch_users).to(DEVICE)
        scores = model.full_user_scores(u_tensor)  # (B, n_items)

        # Mask items already seen in training (no leakage)
        for bi, u in enumerate(batch_users):
            seen = user_seen_items.get(u, set())
            if seen:
                scores[bi, list(seen)] = -float("inf")

        # Top-K predictions
        max_k = max(k_list)
        _, topk = torch.topk(scores, max_k, dim=1)
        topk = topk.cpu().numpy()

        for bi, u in enumerate(batch_users):
            truth = user_positives[u]
            ranked = topk[bi]
            for k in k_list:
                hits = [1 if ranked[j] in truth else 0 for j in range(k)]
                # Recall@K
                metrics[f"recall@{k}"].append(sum(hits) / len(truth))
                # NDCG@K
                dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
                idcg = sum(1 / np.log2(j + 2) for j in range(min(len(truth), k)))
                metrics[f"ndcg@{k}"].append(dcg / idcg if idcg > 0 else 0)

    return {m: np.mean(v) for m, v in metrics.items()}

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. TRAIN + EVAL
# ══════════════════════════════════════════════════════════════
best_val_ndcg = 0.0
for epoch in range(1, CONFIG["n_epochs"] + 1):
    loss = train_one_epoch(epoch)
    val_metrics = evaluate(val_df)
    print(f"\nEpoch {epoch} | loss={loss:.4f} | " +
          " ".join(f"{k}={v:.4f}" for k, v in val_metrics.items()))

    if val_metrics["ndcg@10"] > best_val_ndcg:
        best_val_ndcg = val_metrics["ndcg@10"]
        torch.save(model.state_dict(), "temporal_bpr_mf_best.pt")
        print(f"  ↑ new best val NDCG@10 — saved")

Epoch 1:   0%|          | 0/72 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch 1: 100%|██████████| 72/72 [00:03<00:00, 18.41it/s, loss=0.1207]



Epoch 1 | loss=0.1196 | recall@5=0.0326 recall@10=0.0601 recall@20=0.0948 ndcg@5=0.0330 ndcg@10=0.0475 ndcg@20=0.0621
  ↑ new best val NDCG@10 — saved


Epoch 2: 100%|██████████| 72/72 [00:04<00:00, 17.61it/s, loss=0.1060]



Epoch 2 | loss=0.1134 | recall@5=0.0386 recall@10=0.0641 recall@20=0.0974 ndcg@5=0.0386 ndcg@10=0.0521 ndcg@20=0.0660
  ↑ new best val NDCG@10 — saved


Epoch 3: 100%|██████████| 72/72 [00:03<00:00, 20.85it/s, loss=0.0925]


...


Epoch 8 | loss=0.0732 | recall@5=0.0364 recall@10=0.0625 recall@20=0.0992 ndcg@5=0.0393 ndcg@10=0.0531 ndcg@20=0.0685


Epoch 9: 100%|██████████| 72/72 [00:04<00:00, 14.42it/s, loss=0.0719]



Epoch 9 | loss=0.0724 | recall@5=0.0369 recall@10=0.0626 recall@20=0.0982 ndcg@5=0.0396 ndcg@10=0.0531 ndcg@20=0.0680


Epoch 10: 100%|██████████| 72/72 [00:03<00:00, 20.76it/s, loss=0.0706]



Epoch 10 | loss=0.0717 | recall@5=0.0363 recall@10=0.0634 recall@20=0.0978 ndcg@5=0.0391 ndcg@10=0.0535 ndcg@20=0.0679


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. FINAL TEST EVALUATION
# ══════════════════════════════════════════════════════════════
model.load_state_dict(torch.load("temporal_bpr_mf_best.pt"))
test_metrics = evaluate(test_df)
print("\n══════════════════════════════")
print("TEST SET RESULTS")
print("══════════════════════════════")
for k, v in test_metrics.items():
    print(f"  {k:12s} = {v:.4f}")


══════════════════════════════
TEST SET RESULTS
══════════════════════════════
  recall@5     = 0.0397
  recall@10    = 0.0670
  recall@20    = 0.0945
  ndcg@5       = 0.0421
  ndcg@10      = 0.0565
  ndcg@20      = 0.0680
